## load_climate_normals
Loads the NOAA Climate Normals **station** dataset from the normalize step's Parquet output `RAW_CLIMATE_NORMALS_PROCESSED` (`/Volumes/{CATALOG}/raw/climate_normals/_processed/normals_stations/`) into the all-STRING Bronze table `{BRONZE}.climate_normals_stations`. Keeps the **18** selected columns (5 identity + 13 measure NORMAL values); the `comp_flag_`/`years_` QC companions are dropped (no reporting use; they remain in `_processed`). Source measure names are hyphenated (`ANN-TAVG-NORMAL`) and renamed to underscores here.

**Write strategy (A):** MERGE upsert on `station` (one row per GHCN station). Annual single-vintage snapshot; re-running is a no-op, a new vintage updates in place.

**No archiving (deliberate deviation, recorded per SS18):** annual full snapshot, re-processed each cycle. DDL: `libs/ddl/weather_data.py`. Design: `_dev_planning/design_docs/weather_bronze_load_design.md`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: BRONZE, AUDIT, RAW_CLIMATE_NORMALS_PROCESSED, PIPELINE_RUN_ID,
# STATUS_*, StepLog, Utils, ingestion_log_insert, spark, dbutils, F. The selected columns
# come from climate_normals_columns (single source of truth, shared with the normalize step).
from climate_normals_columns import IDENTITY_COLS, MEASURE_COLS

STEP_SEQUENCE = 2                                   # position owned by the orchestrator
SOURCE_SYSTEM = "climate_normals"
SOURCE_PATH   = RAW_CLIMATE_NORMALS_PROCESSED       # _processed/normals_stations/ (Parquet)
TARGET_TABLE  = f"{BRONZE}.climate_normals_stations"

# The 18 required source columns (5 identity + 13 measures), in the source's hyphenated UPPER
# form as written in the _processed Parquet. The comp_flag_/years_ companions are NOT selected
# (dropped per the design). cell 4 validates presence, cell 5 selects + renames.
REQUIRED_SOURCE_COLS = list(IDENTITY_COLS) + list(MEASURE_COLS)

# Source names carry hyphens (ANN-TAVG-NORMAL) — not legal SQL identifiers. The Bronze column
# names lowercase them and swap hyphens for underscores (matches weather_data.py).
def to_bronze_name(source_col):
    return source_col.lower().replace("-", "_")

# MERGE natural key — one row per GHCN station.
MERGE_KEYS = ["station"]

In [ ]:
# Open the pipeline_step_log row (RUNNING). Closed explicitly in cell 6 (succeed) or by
# step.fail(e) in any work cell's 2-line handler.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
)
print(f"load_climate_normals: step_log_id={step.step_log_id}")

In [ ]:
# Validate the 18 required columns are present in the _processed Parquet BEFORE the bulk
# read. Mirrors load_fhfa: the no-data CHECK stays inside the try; the early EXIT goes OUTSIDE
# (dbutils.notebook.exit() raises an exception that `except Exception` would swallow — SS10.1).
no_data = False
try:
    files = [f.path for f in dbutils.fs.ls(SOURCE_PATH) if f.path.lower().endswith(".parquet")]
    no_data = not files
    if not no_data:
        available_cols = spark.read.parquet(SOURCE_PATH).limit(0).columns
        missing = [c for c in REQUIRED_SOURCE_COLS if c not in available_cols]
        if missing:
            raise ValueError(
                f"[{TARGET_TABLE}] Required column(s) missing in {SOURCE_PATH}:\n  {missing}"
            )
        print(f"load_climate_normals: {len(files)} parquet file(s); schema validated.")
except Exception as e:
    step.fail(e); raise

if no_data:
    step.no_files()
    dbutils.notebook.exit(f"No Parquet files found at {SOURCE_PATH}")

In [ ]:
# Read the _processed Parquet (self-describing, all-STRING). Select the 18 required columns,
# backtick-quoting the hyphenated source names and aliasing to the Bronze (underscore) names;
# the companions are not selected. Add the audit columns (run_id as STRING per SS18).
try:
    selected_cols = [
        F.col(f"`{source_col}`").alias(to_bronze_name(source_col))
        for source_col in REQUIRED_SOURCE_COLS
    ]
    shaped_df = (
        spark.read.parquet(SOURCE_PATH)
            .withColumn("source_file_path", F.col("_metadata.file_path"))
            .select(*selected_cols, "source_file_path")
            .withColumn("inserted_ts", F.current_timestamp())
            .withColumn("run_id", F.lit(PIPELINE_RUN_ID))
    )
    rows_read = shaped_df.count()
    step.rows_read = rows_read
    print(f"load_climate_normals: read {rows_read:,} rows from {SOURCE_PATH}")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Strategy A: MERGE upsert on the natural key. INSERT */UPDATE SET * match by column name.
# MERGE metrics (num_inserted_rows / num_updated_rows) come back as the result row on
# Databricks [Projected — confirm on first run]; fall back to the post-pre delta if absent.
try:
    shaped_df.createOrReplaceTempView("climate_normals_staging")
    on_clause = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEYS)

    pre_count = spark.table(TARGET_TABLE).count()
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS t
        USING climate_normals_staging AS s
        ON {on_clause}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first().asDict()
    post_count = spark.table(TARGET_TABLE).count()

    inserted = metrics.get("num_inserted_rows")
    updated  = metrics.get("num_updated_rows")
    if inserted is None:
        inserted = post_count - pre_count
    if post_count - pre_count != inserted:
        raise AssertionError(
            f"[{TARGET_TABLE}] Insert-count mismatch: MERGE reported {inserted:,} inserts, "
            f"but row count grew by {post_count - pre_count:,} "
            f"(pre {pre_count:,}, post {post_count:,})."
        )

    step.rows_written = inserted
    step.succeed()
    print(f"load_climate_normals: MERGE done — inserted={inserted:,} updated={updated} "
          f"(read={rows_read:,}, pre={pre_count:,}, post={post_count:,}).")
except Exception as e:
    step.fail(e); raise

# ingestion_log (leaf tier) runs AFTER succeed() and OUTSIDE the write try.
files_df = shaped_df.select("source_file_path").distinct()
res = ingestion_log_insert(
    spark, AUDIT, files_df, PIPELINE_RUN_ID, step.step_log_id,
    source_system=SOURCE_SYSTEM, target_table=TARGET_TABLE,
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"load_climate_normals: WARNING ingestion_log insert failed: {res['error_message']}")